# 04 — Trực quan hóa Self-Attention của ViT-B/16 trên Oxford-IIIT Pet

Notebook này trực quan hóa cơ chế tập trung thị giác của mô hình **ViT-B/16** đã fine-tune (thí nghiệm **E4**, checkpoint `best.pt`) trên tập `test` của **Oxford-IIIT Pet** (37 giống chó và mèo):
1. **Attention Map của token `[CLS]` theo từng Encoder Layer** (12 tầng `encoder_layer_0` → `encoder_layer_11`, trung bình qua 12 attention heads).
2. **Attention Rollout** (*Abnar & Zuidema, ACL 2020* — "Quantifying Attention Flow in Transformers") có tính đến kết nối thặng dư (residual connection $A + I$) và chiến lược tổng hợp qua các head (`mean` / `max` / `min` kèm ngưỡng loại bỏ nhiễu `discard_ratio`).
3. **So sánh trực quan trên các ca dự đoán ĐÚNG và dự đoán SAI** trên tập `test`, tự động lưu toàn bộ hình ảnh phân tích ra thư mục `figures/attention/`.

In [ ]:
import os
import sys
from pathlib import Path

# Ho tro chay ca tren Google Colab lan may cuc bo
if "google.colab" in sys.modules:
    repo_dir = Path("/content/vit-oxford-pets")
    if not repo_dir.exists():
        !git clone -q https://github.com/duyenphung206-sudo/vit-oxford-pets.git /content/vit-oxford-pets
    os.chdir(repo_dir)
    !pip install -q -r requirements.txt
    DATA_ROOT = "/content/data"
    CHECKPOINT_PATH = "/content/drive/MyDrive/vit-oxford-pets/checkpoints/e4_vit_full_ft/seed44/best.pt"
else:
    DATA_ROOT = "data"
    CHECKPOINT_PATH = "checkpoints/e4_vit_full_ft/seed44/best.pt"

FIG_DIR = Path("figures/attention")
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Working directory:", Path.cwd())
print("Dataset root:", DATA_ROOT)
print("Checkpoint path:", CHECKPOINT_PATH)
print("Output figure dir:", FIG_DIR)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageOps
from torchvision import models
from torchvision.datasets import OxfordIIITPet

from src.data import class_names, read_annotations
from src.models.factory import build_model
from src.utils import get_device, load_config

device = get_device()
print("Device:", device)

## 1. Nạp mô hình E4 (`ViT-B/16` Fine-tuned) & dữ liệu kiểm thử

- Nạp cấu hình `configs/e4_vit_full_ft.yaml` và trọng số checkpoint `best.pt` (hỗ trợ cả raw `state_dict` từ `src/train.py` lẫn dict có khóa `"model"`).
- Nếu chưa tải checkpoint thật về ổ đĩa, có thể gán `ALLOW_PRETRAINED_FALLBACK = True` để kiểm thử luồng trích xuất attention trên backbone ImageNet-1K.

In [ ]:
CONFIG_PATH = "configs/e4_vit_full_ft.yaml"
ALLOW_PRETRAINED_FALLBACK = False  # Doi thanh True neu chi muon test pipeline khi chua co best.pt

cfg = load_config(CONFIG_PATH)
weights = models.ViT_B_16_Weights.IMAGENET1K_V1
preprocess = weights.transforms()

ckpt_file = Path(CHECKPOINT_PATH)
if ckpt_file.exists():
    model_cfg = {**cfg["model"], "pretrained": False}
    model = build_model(model_cfg)
    ckpt = torch.load(ckpt_file, map_location="cpu", weights_only=True)
    state_dict = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
    model.load_state_dict(state_dict)
    model_tag = f"{cfg['experiment']} ({ckpt_file.parent.name}/{ckpt_file.name})"
    print("Da nap checkpoint thanh cong:", ckpt_file)
elif ALLOW_PRETRAINED_FALLBACK:
    model = build_model(cfg["model"])
    model_tag = "vit_b_16_imagenet_fallback"
    print("Canh bao: Khong tim thay checkpoint, dang dung trong so pretrained fallback.")
else:
    raise FileNotFoundError(
        f"Khong tim thay checkpoint tai '{ckpt_file}'. "
        "Hay cap nhat CHECKPOINT_PATH toi file best.pt cua E4 (seed 44) hoac bat ALLOW_PRETRAINED_FALLBACK=True."
    )

model = model.to(device).eval()
dataset = OxfordIIITPet(root=DATA_ROOT, split="test", download=True)
pet_classes = class_names(read_annotations(DATA_ROOT, "trainval"))
assert len(pet_classes) == 37
print(f"Model: {model_tag} | Test samples: {len(dataset)} | Classes: {len(pet_classes)}")

## 2. Trích xuất Attention Weights theo từng Layer & thuật toán Attention Rollout

### Cơ sở toán học
Với ảnh đầu vào $224 \times 224$ và kích thước patch $P = 16$, ViT-B/16 chia ảnh thành lưới $14 \times 14 = 196$ patch tokens cộng thêm $1$ token `[CLS]` ở vị trí $0$, tạo thành chuỗi $N = 197$ tokens đi qua $L = 12$ khối Encoder (`12` heads mỗi khối).

1. **`[CLS]` Attention theo từng tầng $l \in \{1, \dots, 12\}$**:
   Trung bình ma trận attention qua $H = 12$ heads:
   $$\bar{A}^{(l)} = \frac{1}{H} \sum_{h=1}^{H} A^{(l, h)} \in \mathbb{R}^{197 \times 197}$$
   Hàng đầu tiên $\bar{A}^{(l)}_{0, 1:197} \in \mathbb{R}^{196}$ đo mức độ token `[CLS]` chú ý trực tiếp tới từng patch không gian tại tầng $l$, sau đó được reshape thành lưới $14 \times 14$ và nội suy song tuyến tính (bilinear) lên $224 \times 224$.

2. **Attention Rollout (*Abnar & Zuidema, 2020*)**:
   Do mỗi khối Transformer có kết nối thặng dư (residual connection) $X^{(l)} = X^{(l-1)} + \text{MSA}(X^{(l-1)})$, luồng thông tin truyền qua từng tầng được mô hình hóa bằng ma trận chuyển tiếp:
   $$\tilde{A}^{(l)} = 0.5 \, \hat{A}^{(l)} + 0.5 \, I_{197}, \qquad \text{chuẩn hóa từng hàng sao cho } \sum_{j} \tilde{A}^{(l)}_{i,j} = 1$$
   trong đó $\hat{A}^{(l)}$ là ma trận attention sau khi tổng hợp các head (`mean` hoặc `max`) và loại bỏ các trọng số nhiễu nhỏ nhất theo tỷ lệ `discard_ratio` (giữ nguyên cột `[CLS]`).
   Nhân tích lũy từ tầng đầu đến tầng cuối:
   $$R = \tilde{A}^{(L)} \tilde{A}^{(L-1)} \cdots \tilde{A}^{(1)}$$
   Vector $R_{0, 1:197}$ biểu diễn **luồng chú ý tổng hợp (attention flow)** từ 196 patch ảnh đầu vào hội tụ về token `[CLS]` ở đầu ra cuối cùng.

In [ ]:
class ViTAttentionExtractor:
    """Bat trọn attention weights (B, num_heads, 197, 197) cua 12 encoder blocks trong torchvision ViT."""

    def __init__(self, vit_model: nn.Module) -> None:
        self.model = vit_model
        self.attentions: list[torch.Tensor] = []
        self._handles: list[torch.utils.hooks.RemovableHandle] = []

    def _pre_hook(self, module: nn.Module, args: tuple, kwargs: dict):
        # Torchvision mac dinh goi self_attention(..., need_weights=False)
        kwargs["need_weights"] = True
        kwargs["average_attn_weights"] = False
        return args, kwargs

    def _forward_hook(self, module: nn.Module, inputs: tuple, outputs: tuple) -> None:
        attn_weights = outputs[1]
        if attn_weights is not None:
            self.attentions.append(attn_weights.detach().float().cpu())

    def __enter__(self) -> "ViTAttentionExtractor":
        self.attentions.clear()
        for block in self.model.encoder.layers:
            self._handles.append(block.self_attention.register_forward_pre_hook(self._pre_hook, with_kwargs=True))
            self._handles.append(block.self_attention.register_forward_hook(self._forward_hook))
        return self

    def __exit__(self, exc_type, exc_val, exc_tb) -> None:
        for h in self._handles:
            h.remove()
        self._handles.clear()

    @torch.inference_mode()
    def extract(self, input_tensor: torch.Tensor) -> tuple[torch.Tensor, list[torch.Tensor]]:
        self.attentions.clear()
        logits = self.model(input_tensor)
        if len(self.attentions) != len(self.model.encoder.layers):
            raise RuntimeError(
                f"Ky vong {len(self.model.encoder.layers)} ma tran attention, nhan duoc {len(self.attentions)}."
            )
        return logits, list(self.attentions)


def attention_to_grid(cls_attn_1d: torch.Tensor, image_size: int = 224) -> np.ndarray:
    """Chuyen vector 196 patch weights thanh ban do nhiet (image_size, image_size) chuan hoa [0, 1]."""
    num_patches = cls_attn_1d.numel()
    grid_size = int(round(num_patches ** 0.5))
    assert grid_size * grid_size == num_patches, f"So patch {num_patches} khong phai so chinh phuong."
    grid = cls_attn_1d.reshape(1, 1, grid_size, grid_size)
    upsampled = F.interpolate(grid, size=(image_size, image_size), mode="bilinear", align_corners=False)[0, 0]
    arr = upsampled.numpy()
    arr = arr - arr.min()
    if arr.max() > 0:
        arr = arr / arr.max()
    return arr


def compute_layer_cls_maps(attentions: list[torch.Tensor], image_size: int = 224) -> list[np.ndarray]:
    """Tinh [CLS] attention map trung binh qua 12 heads cho tung layer."""
    maps = []
    for layer_attn in attentions:
        # layer_attn: (1, 12, 197, 197)
        mean_heads = layer_attn[0].mean(dim=0)  # (197, 197)
        cls_to_patches = mean_heads[0, 1:]      # (196,)
        maps.append(attention_to_grid(cls_to_patches, image_size=image_size))
    return maps


def compute_attention_rollout(
    attentions: list[torch.Tensor],
    head_fusion: str = "mean",
    discard_ratio: float = 0.0,
    image_size: int = 224,
) -> np.ndarray:
    """Tinh Attention Rollout (Abnar & Zuidema, 2020) cho token [CLS]."""
    num_tokens = attentions[0].size(-1)
    eye = torch.eye(num_tokens)
    rollout = eye.clone()

    for layer_attn in attentions:
        attn = layer_attn[0]  # (num_heads, 197, 197)
        if head_fusion == "mean":
            fused = attn.mean(dim=0)
        elif head_fusion == "max":
            fused = attn.max(dim=0).values
        elif head_fusion == "min":
            fused = attn.min(dim=0).values
        else:
            raise ValueError(f"head_fusion khong hop le: {head_fusion}")

        if discard_ratio > 0.0:
            # Loai bo cac trong so nho nhat o tung hang (giu nguyen token [CLS] o cot 0)
            patch_part = fused[:, 1:]
            k_drop = int(patch_part.size(1) * discard_ratio)
            if k_drop > 0:
                thresh = torch.kthvalue(patch_part, k_drop, dim=1, keepdim=True).values
                patch_part = torch.where(patch_part <= thresh, torch.zeros_like(patch_part), patch_part)
                fused = torch.cat([fused[:, :1], patch_part], dim=1)

        # Cong ket noi thang du (A + I) va chuan hoa tong hang = 1
        aug_attn = 0.5 * fused + 0.5 * eye
        aug_attn = aug_attn / aug_attn.sum(dim=-1, keepdim=True).clamp(min=1e-8)
        rollout = aug_attn @ rollout

    cls_rollout = rollout[0, 1:]
    return attention_to_grid(cls_rollout, image_size=image_size)

## 3. Quét tập `test` để chọn các ca dự đoán ĐÚNG và dự đoán SAI

Chạy suy luận trên toàn bộ 3.669 ảnh của tập `test` để chọn:
- **4 ảnh dự đoán ĐÚNG** (đại diện cả mèo và chó, độ tự tin cao).
- **4 ảnh dự đoán SAI** (các trường hợp nhầm lẫn giống gần nhau có xác suất dự đoán sai cao nhất để phân tích lỗi).

In [ ]:
from torch.utils.data import DataLoader
from src.data import build_dataloaders

loaders = build_dataloaders(
    data_dir=DATA_ROOT,
    split_dir=cfg["data"]["splits_dir"],
    batch_size=64,
    num_workers=2 if torch.cuda.is_available() else 0,
    seed=cfg["train"]["seed"],
    pin_memory=torch.cuda.is_available(),
    augment=False,
)
test_loader = loaders["test"]
test_rows = test_loader.dataset.rows

all_probs = []
all_targets = []
with torch.inference_mode():
    for images, labels in test_loader:
        logits = model(images.to(device, non_blocking=True))
        probs = logits.float().softmax(dim=-1).cpu()
        all_probs.append(probs)
        all_targets.append(labels)

probs_tensor = torch.cat(all_probs, dim=0)
targets_tensor = torch.cat(all_targets, dim=0)
pred_probs, pred_indices = probs_tensor.max(dim=1)

correct_mask = pred_indices == targets_tensor
test_acc = correct_mask.float().mean().item()
print(f"Test Top-1 Accuracy: {test_acc * 100:.2f}% ({correct_mask.sum().item()}/{len(targets_tensor)})")

# Chon 4 mau DUNG (2 meo + 2 cho, do tu tin cao, khac giong)
correct_candidates = []
for idx in torch.where(correct_mask)[0].tolist():
    true_id = int(targets_tensor[idx])
    breed = pet_classes[true_id]
    is_cat = breed[0].isupper()  # Quy uoc Oxford-IIIT Pet: ten giong meo viet hoa chu cai dau
    correct_candidates.append((float(pred_probs[idx]), idx, breed, is_cat))
correct_candidates.sort(reverse=True)

selected_correct = []
used_breeds = set()
for target_is_cat in (True, False):
    count = 0
    for conf, idx, breed, is_cat in correct_candidates:
        if is_cat == target_is_cat and breed not in used_breeds:
            used_breeds.add(breed)
            selected_correct.append(idx)
            count += 1
            if count == 2:
                break

# Chon 4 mau SAI (do tu tin du doan sai cao nhat, khac cap nham lan)
wrong_candidates = []
for idx in torch.where(~correct_mask)[0].tolist():
    true_id = int(targets_tensor[idx])
    pred_id = int(pred_indices[idx])
    wrong_candidates.append((float(pred_probs[idx]), idx, pet_classes[true_id], pet_classes[pred_id]))
wrong_candidates.sort(reverse=True)

selected_wrong = []
used_pairs = set()
for conf, idx, true_b, pred_b in wrong_candidates:
    pair = (true_b, pred_b)
    if pair not in used_pairs:
        used_pairs.add(pair)
        selected_wrong.append(idx)
        if len(selected_wrong) == 4:
            break

def load_sample_detail(idx: int, status: str) -> dict:
    row = test_rows[idx]
    img_path = Path(DATA_ROOT) / "oxford-iiit-pet" / "images" / f"{row['id']}.jpg"
    with Image.open(img_path) as im:
        orig_img = ImageOps.exif_transpose(im).convert("RGB")
    # Resize 256 -> CenterCrop 224 de khop chinh xac he toa do patch 14x14
    vis_img = orig_img.resize((256, 256), Image.Resampling.BILINEAR)
    left = (256 - 224) // 2
    vis_img = vis_img.crop((left, left, left + 224, left + 224))

    input_tensor = preprocess(orig_img).unsqueeze(0).to(device)
    with ViTAttentionExtractor(model) as extractor:
        logits, attentions = extractor.extract(input_tensor)
    probs = logits.float().softmax(dim=-1)[0].cpu()
    top3_vals, top3_ids = probs.topk(3)
    true_label = pet_classes[row["label"]]
    pred_label = pet_classes[int(top3_ids[0])]

    return {
        "status": status,
        "index": idx,
        "image_id": row["id"],
        "vis_image": np.asarray(vis_img),
        "true_label": true_label,
        "pred_label": pred_label,
        "pred_prob": float(top3_vals[0]),
        "true_prob": float(probs[row["label"]]),
        "top3": [(pet_classes[int(i)], float(v)) for i, v in zip(top3_ids.tolist(), top3_vals.tolist())],
        "layer_maps": compute_layer_cls_maps(attentions, image_size=224),
        "rollout_mean": compute_attention_rollout(attentions, head_fusion="mean", discard_ratio=0.0),
        "rollout_max_pruned": compute_attention_rollout(attentions, head_fusion="max", discard_ratio=0.5),
    }

correct_samples = [load_sample_detail(i, "CORRECT") for i in selected_correct]
wrong_samples = [load_sample_detail(i, "WRONG") for i in selected_wrong]

summary_rows = []
for s in correct_samples + wrong_samples:
    summary_rows.append({
        "Status": s["status"],
        "Image ID": s["image_id"],
        "True Breed": s["true_label"],
        "Predicted Breed": f"{s['pred_label']} ({s['pred_prob']*100:.1f}%)",
        "True Class Prob": f"{s['true_prob']*100:.1f}%",
        "Top-2 / Top-3": ", ".join(f"{b} ({p*100:.1f}%)" for b, p in s["top3"][1:]),
    })
pd.DataFrame(summary_rows)

## 4. Trực quan hóa `[CLS]` Attention Map theo từng Layer (Layer 1 → Layer 12)

Quan sát cách token `[CLS]` dịch chuyển vùng chú ý từ **đặc trưng cục bộ tần số thấp** ở các tầng nông (Layer 1–4) sang **đặc trưng ngữ nghĩa bộ phận cơ thể** (mắt, tai, mõm, hoa văn lông) ở các tầng sâu (Layer 9–12).

In [ ]:
def plot_layerwise_cls_attention(sample: dict, out_path: Path) -> None:
    fig, axes = plt.subplots(2, 7, figsize=(18, 5.8))
    axes = axes.flatten()

    # O dau tien: anh goc 224x224
    axes[0].imshow(sample["vis_image"])
    axes[0].set_title(
        f"[{sample['status']}] {sample['image_id']}\n"
        f"True: {sample['true_label']}\n"
        f"Pred: {sample['pred_label']} ({sample['pred_prob']*100:.1f}%)",
        fontsize=8.5,
        fontweight="bold",
        color="#15803d" if sample["status"] == "CORRECT" else "#b91c1c",
    )
    axes[0].axis("off")

    # 12 o tiep theo: [CLS] attention cua Layer 1..12
    for layer_idx in range(12):
        ax = axes[layer_idx + 1]
        ax.imshow(sample["vis_image"])
        ax.imshow(sample["layer_maps"][layer_idx], cmap="jet", alpha=0.52, vmin=0.0, vmax=1.0)
        ax.set_title(f"Layer {layer_idx + 1}", fontsize=9)
        ax.axis("off")

    # O cuoi cung: Attention Rollout tong hop
    axes[13].imshow(sample["vis_image"])
    axes[13].imshow(sample["rollout_mean"], cmap="jet", alpha=0.52, vmin=0.0, vmax=1.0)
    axes[13].set_title("Rollout (Mean)", fontsize=9, fontweight="bold")
    axes[13].axis("off")

    fig.suptitle(
        f"[CLS] Attention theo tung Encoder Layer — {sample['image_id']} ({sample['status']})",
        fontsize=12,
        fontweight="bold",
    )
    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)

# Ve chi tiet 12 layer cho 2 mau tieu bieu (1 dung, 1 sai)
plot_layerwise_cls_attention(correct_samples[0], FIG_DIR / f"layerwise_correct_{correct_samples[0]['image_id']}.png")
plot_layerwise_cls_attention(wrong_samples[0], FIG_DIR / f"layerwise_wrong_{wrong_samples[0]['image_id']}.png")

## 5. So sánh Attention Rollout (Abnar & Zuidema, 2020) giữa ảnh ĐÚNG và ảnh SAI

Mỗi hàng hiển thị một ảnh mẫu với 4 góc nhìn:
1. **Ảnh đầu vào (`224 × 224`)**
2. **`[CLS]` Attention tầng cuối (Layer 12)**
3. **Attention Rollout (`head_fusion='mean'`, `discard_ratio=0.0`)**
4. **Attention Rollout (`head_fusion='max'`, `discard_ratio=0.5`)** — làm nổi bật vùng đặc trưng phân biệt cốt lõi sau khi lọc 50% trọng số nhiễu nền.

In [ ]:
def plot_rollout_comparison(samples: list[dict], group_title: str, out_path: Path) -> None:
    n = len(samples)
    fig, axes = plt.subplots(n, 4, figsize=(13.5, 3.3 * n))
    if n == 1:
        axes = axes[None, :]

    col_titles = [
        "Anh goc (224x224)",
        "[CLS] Layer 12 (Mean Heads)",
        "Attention Rollout (Mean)",
        "Attention Rollout (Max, Drop 50%)",
    ]

    for r, s in enumerate(samples):
        status_color = "#15803d" if s["status"] == "CORRECT" else "#b91c1c"
        axes[r, 0].imshow(s["vis_image"])
        axes[r, 0].set_ylabel(
            f"{s['image_id']}\nTrue: {s['true_label']}\nPred: {s['pred_label']} ({s['pred_prob']*100:.1f}%)",
            fontsize=8.5,
            fontweight="bold",
            color=status_color,
        )

        heatmaps = [s["layer_maps"][-1], s["rollout_mean"], s["rollout_max_pruned"]]
        for c, hmap in enumerate(heatmaps, start=1):
            axes[r, c].imshow(s["vis_image"])
            axes[r, c].imshow(hmap, cmap="jet", alpha=0.52, vmin=0.0, vmax=1.0)

        for c in range(4):
            if r == 0:
                axes[r, c].set_title(col_titles[c], fontsize=10, fontweight="bold")
            axes[r, c].set_xticks([])
            axes[r, c].set_yticks([])

    fig.suptitle(group_title, fontsize=13, fontweight="bold", y=1.01)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)

plot_rollout_comparison(
    correct_samples,
    "Attention Rollout — Cac ca du doan DUNG (Correct Predictions)",
    FIG_DIR / "rollout_correct_grid.png",
)
plot_rollout_comparison(
    wrong_samples,
    "Attention Rollout — Cac ca du doan SAI (Misclassified Cases)",
    FIG_DIR / "rollout_wrong_grid.png",
)

In [ ]:
# Liet ke cac file hinh da xuat ra figures/attention/
saved_files = sorted(FIG_DIR.glob("*.png"))
print(f"Da luu {len(saved_files)} hinh vao '{FIG_DIR}':")
for f in saved_files:
    print(f" - {f.name} ({f.stat().st_size / 1024:.1f} KB)")

## 6. Phân tích & Nhận xét từ bản đồ Attention

1. **Diễn biến `[CLS]` Attention qua chiều sâu mạng (Layer 1 → Layer 12)**:
   - **Các tầng nông (Layer 1–4)**: Bản đồ attention phân tán rộng trên cả nền ảnh, đường biên vật thể và các kết cấu lặp lại do các head chủ yếu thực hiện tổng hợp ngữ cảnh toàn cục sớm.
   - **Các tầng trung gian (Layer 5–8)**: Vùng kích hoạt bắt đầu thu hẹp dần về khối cơ thể con vật, loại bỏ phần lớn phông nền.
   - **Các tầng sâu (Layer 9–12)**: `[CLS]` tập trung sắc nét vào các vùng đặc trưng phân loại mịn (fine-grained cues) như **khuôn mặt, hình dáng tai, sống mũi/mõm và hoa văn bộ lông**.

2. **So sánh `[CLS]` Layer cuối với Attention Rollout (*Abnar & Zuidema, 2020*)**:
   - Bản đồ `[CLS]` đơn lẻ tại Layer 12 chỉ phản ánh trọng số trực tiếp ở bước cuối cùng, đôi khi bỏ sót các patch trung gian đã truyền thông tin qua kết nối thặng dư (residual connections).
   - **Attention Rollout** tích hợp đường truyền thông tin qua toàn bộ 12 tầng với ma trận $0.5 A + 0.5 I$, cho bản đồ bao phủ đầy đủ và mượt mà hơn trên toàn bộ chủ thể thú cưng.

3. **Khác biệt giữa ca dự đoán ĐÚNG và ca dự đoán SAI**:
   - **Ca dự đoán ĐÚNG**: Vùng chú ý cao nhất (màu đỏ/cam trên heatmap) bám sát vùng đầu và đặc điểm nhận diện đặc trưng của giống.
   - **Ca dự đoán SAI**: Thường rơi vào các nhóm giống có ngoại hình rất gần nhau (ví dụ nhóm chó sục/chó săn tai cụp hoặc mèo lông ngắn) hoặc các ảnh có góc chụp khuất mặt, ánh sáng gắt, vật thể nền gây nhiễu khiến vùng chú ý của mô hình bị phân tán sang phông nền hoặc phần thân thiếu đặc trưng.